In [12]:
import gc
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# 1. Define explicit file paths from the scratchpad directory
path_loans_lenders = r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\loans_lenders.csv"

# Load the source datasets into memory
df_loans_lenders = pd.read_csv(path_loans_lenders)

In [13]:
print(df_loans_lenders.shape)
print(df_loans_lenders.columns)
print(df_loans_lenders.info())
df_loans_lenders.head(50)

(1387432, 2)
Index(['loan_id', 'lenders'], dtype='str')
<class 'pandas.DataFrame'>
RangeIndex: 1387432 entries, 0 to 1387431
Data columns (total 2 columns):
 #   Column   Non-Null Count    Dtype
---  ------   --------------    -----
 0   loan_id  1387432 non-null  int64
 1   lenders  1387432 non-null  str  
dtypes: int64(1), str(1)
memory usage: 346.3 MB
None


,loan_id,lenders
0,483693,"muc888, sam4326, camaran3922, lachheb1865, reb..."
1,483738,"muc888, nora3555, williammanashi, barbara5610,..."
2,485000,"muc888, terrystl, richardandsusan8352, sherri4..."
3,486087,"muc888, james5068, rudi5955, daniel9859, don92..."
4,534428,"muc888, niki3008, teresa9174, mike4896, david7..."
5,558112,"muc888, tristan7990, shivaun4955, sam44598568,..."
6,563395,"muc888, john38425073, trolltech4460, marianne8..."
7,575414,"muc888, dougal1825, dougal1825, jensdamsgaardv..."
8,578029,"muc888, rebecca3038, paul1853, paul1853, paul1..."
9,551251,"klaus5005, john70242429, john70242429, terry93..."


In [14]:
# Gather all structural metrics
missing_count = df_loans_lenders.isnull().sum()
missing_percent = (df_loans_lenders.isnull().sum() / len(df_loans_lenders)) * 100
column_types = df_loans_lenders.dtypes

# Combine them into a single DataFrame
full_report_loans_lenders = pd.DataFrame({
    "Data Type": column_types,
    "Missing Count": missing_count,
    "Missing %": missing_percent
})


# Note: We do not sort by "Missing %" here so that the columns 
# stay in their original order (0 to 102), making slicing accurate.

# Columns 1 to 25
print(full_report_loans_lenders.iloc[0:25])

# Strip hidden spaces from categorical string columns
loans_lenders_cols = ['loan_id', 'lenders']

        Data Type  Missing Count  Missing %
loan_id     int64              0        0.0
lenders       str              0        0.0


In [15]:
# Check for missing coordinate rows
missing_id = df_loans_lenders["loan_id"].isna().sum()
missing_lenders = df_loans_lenders["lenders"].isna().sum()
print(f"\nMissing Data -> ID: {missing_id}, Lenders: {missing_lenders}")

# Initialize a clean structural copy
df_loans_lenders_clean = df_loans_lenders.copy()

# Standardize column header strings
df_loans_lenders_clean.columns = df_loans_lenders_clean.columns.str.strip().str.lower()

# Deduplicate based strictly on the unique primary identifier
df_loans_lenders_clean = df_loans_lenders_clean.drop_duplicates(subset=['loan_id'])


Missing Data -> ID: 0, Lenders: 0


In [16]:
# 1. Ensure the lenders column is treated as a clean string format
df_loans_lenders_clean = df_loans_lenders.copy()
df_loans_lenders_clean['lenders'] = df_loans_lenders_clean['lenders'].astype(str)

# 2. Convert the comma-separated text strings into true Python lists
df_loans_lenders_clean['lenders'] = df_loans_lenders_clean['lenders'].str.split(',')

# 3. Explode the lists to transform every single username into its own individual row
df_loans_lenders_clean = df_loans_lenders_clean.explode('lenders')

# 4. Strip out any trailing or leading whitespace spaces from the newly separated usernames
df_loans_lenders_clean['lenders'] = df_loans_lenders_clean['lenders'].str.strip()

# 5. Standardize column names to clean lowercase text references
df_loans_lenders_clean.columns = df_loans_lenders_clean.columns.str.strip().str.lower()

# 6. Remove any empty or accidental null entries created by trailing commas
df_loans_lenders_clean = df_loans_lenders_clean.dropna(subset=['lenders'])
df_loans_lenders_clean = df_loans_lenders_clean[df_loans_lenders_clean['lenders'] != '']

# 7. Export the staging file to your local archive folder
df_loans_lenders_clean.to_csv(
    r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_loans_lenders.csv",
    index=False
)

print("=== LOANS LENDERS FLATTENED VERIFICATION ===")
print(f"Expanded Shape: {df_loans_lenders_clean.shape}")
print(df_loans_lenders_clean.head(50))

=== LOANS LENDERS FLATTENED VERIFICATION ===
Expanded Shape: (28293931, 2)
   loan_id               lenders
0   483693                muc888
0   483693               sam4326
0   483693           camaran3922
0   483693           lachheb1865
0   483693           rebecca3499
0   483693         karlheinz4543
0   483693               jerrydb
0   483693             paula8951
0   483693                  gmct
0   483693              amra9383
0   483693                 r3922
0   483693             brian9451
0   483693             shree8053
0   483693              alan5513
0   483693             oisin3389
0   483693             helle8622
0   483693                bo3186
0   483693               ric8947
0   483693        daniel98469874
0   483693              nick9464
0   483693       deborah12671549
0   483693           matthew9831
0   483693              john6330
0   483693              john9479
0   483693          mattiaslaven
0   483693          jonathan2867
0   483693             jason3883
0

In [17]:
print(df_loans_lenders_clean.tail(50))

         loan_id                   lenders
1387429  1206220                 lidia5529
1387429  1206220  advocatesforrepeatbo7716
1387429  1206220                    impact
1387429  1206220                  beth9974
1387429  1206220                  matt9892
1387429  1206220                  paul8567
1387429  1206220                 dborquezm
1387429  1206220                  evab4407
1387429  1206220                 kelly4849
1387429  1206220                  jody6226
1387429  1206220            sandra70423243
1387429  1206220                  rene1144
1387429  1206220           michael14598672
1387429  1206220              alex26799277
1387429  1206220                 jenny3234
1387429  1206220                daniel5141
1387429  1206220             trolltech4460
1387429  1206220                claude3821
1387429  1206220              john56514883
1387429  1206220              bernardo6554
1387429  1206220                 karen8410
1387429  1206220          themissionbeltco
1387429  12

In [18]:
# Check for absolute identical row duplicates (same loan and same lender mapping)
duplicate_mappings = df_loans_lenders_clean.duplicated(subset=['loan_id', 'lenders']).sum()

print("=== FINAL LOANS_LENDERS COMPREHENSIVE QUALITY REPORT ===")
print(f"Total Rows in Memory: {df_loans_lenders_clean.shape[0]}")
print(f"Total Missing Values in Dataframe:\n{df_loans_lenders_clean.isnull().sum()}")
print(f"Exact Duplicate Loan-to-Lender Mapping Pairs found: {duplicate_mappings}")

# Safely drop duplicate pairs only if they exist
if duplicate_mappings > 0:
    df_loans_lenders_clean = df_loans_lenders_clean.drop_duplicates(subset=['loan_id', 'lenders'])
    print(f"Cleaned Row Count after removing duplicate pairs: {df_loans_lenders_clean.shape[0]}")

=== FINAL LOANS_LENDERS COMPREHENSIVE QUALITY REPORT ===
Total Rows in Memory: 28293931
Total Missing Values in Dataframe:
loan_id    0
lenders    0
dtype: int64
Exact Duplicate Loan-to-Lender Mapping Pairs found: 834845
Cleaned Row Count after removing duplicate pairs: 27459086


In [19]:
# Overwrite the staging file with the final deduplicated dataset
df_loans_lenders_clean.to_csv(
    r"C:\Users\sanjay\OneDrive\Desktop\scratchpad_archive\002.kiva_loans_datasets\cleaned_loans_lenders.csv",
    index=False
)

print("Final deduplicated loans_lenders dataset successfully exported!")

Final deduplicated loans_lenders dataset successfully exported!
